# Credential-Sharing Graph — EDA

Exploration only. Any logic worth keeping graduates into `graph.py` / `charts.py` as a named function — this notebook should only ever call into those modules, not reimplement analysis logic inline.

**Setup** (see `../README.md` for the full version): run once per machine —
```
uv sync --group test --group analysis
uv run python -m ipykernel install --user --name bear-trap-analysis --display-name "Python (bear-trap analysis)" --env PYTHONPATH "$(pwd)"
```
then select the "Python (bear-trap analysis)" kernel for this notebook — the same one `fingerprinting/notebooks/eda.ipynb` uses. Both notebooks import via absolute `honeypot.analysis.*` package paths, so one shared kernel (repo root on `PYTHONPATH`) works for both; there's no per-submodule scoping needed anymore.

See `.claude/tasks/active-task.md` for the full spec (local/gitignored — won't exist on a fresh clone).

In [ ]:
import polars as pl

from honeypot.analysis.common.extract import get_sessions
from honeypot.analysis.network.charts import (
    plot_cluster_size_distribution,
    plot_top_clusters_subgraph,
)
from honeypot.analysis.network.graph import assign_clusters, build_credential_graph

## Load data

`"offline"` reads the most recent local snapshot from the shared `../common/outputs/snapshots/` (no GCP credentials needed). Switch to `"live"` to pull fresh from BigQuery and refresh the snapshot as a side effect.

In [ ]:
MODE = "offline"

events = get_sessions(MODE)
events.select("src_ip", "username", "password").head()

## Build the credential-sharing graph

Bipartite graph of `src_ip` nodes and `(username, password)` credential-pair nodes; `assign_clusters` runs `networkx.connected_components` over it to group IPs that ever shared a pair.

By default, `build_credential_graph` excludes a manual denylist of generic credential pairs (`graph._GENERIC_CREDENTIAL_PAIRS`) derived from this same snapshot -- pairs attempted by >=20 distinct IPs, the point where the frequency distribution has a real cliff. Without it, a single pair like `admin`/`admin` acts as a hub node that merges every IP that ever tried it into one cluster regardless of any other relationship -- that's exactly what produced a single 690-IP cluster out of 883 total IPs before this list existed. Pass `exclude_pairs=frozenset()` to see the unfiltered graph instead.

In [ ]:
graph = build_credential_graph(events)
clusters = assign_clusters(graph)

print(f"{graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
print(f"{clusters.height} IPs clustered")
print(f"{clusters['cluster_id'].n_unique()} distinct clusters")
clusters.sort("cluster_size", descending=True).head(10)

In [ ]:
cluster_zero = clusters.filter(pl.col("cluster_id") == 0)
events_cluster_zero = events.filter(
    pl.col("src_ip").is_in(cluster_zero["src_ip"].to_list())
)
events_cluster_zero.select("src_ip", "username", "password").unique()

In [ ]:
events.filter(pl.col("username") == "grok")

## Baseline charts

`charts.py` forces the `Agg` backend (headless-safe for tests/`pipeline.py`), so these calls save a PNG but won't render inline on their own — wrap with `Image(...)` to preview.

In [ ]:
from IPython.display import Image

Image(plot_cluster_size_distribution(clusters))

In [ ]:
path = plot_top_clusters_subgraph(graph)
if path:
    display(Image(path))
else:
    print("Skipped -- see the message above: subgraph too large to plot.")

## Exploration

Ad hoc from here — how much a handful of generic credential pairs dominate the largest clusters (the risk flagged in the module's spec), whether cluster membership correlates with `cloud_provider`/`sensor` (same campaign hitting both clouds), whatever the data suggests.

In [ ]:
clusters.filter(pl.col("cluster_size") == 1).height

## What got excluded: generic credential pairs

The denylisted pairs (`graph._GENERIC_CREDENTIAL_PAIRS`) aren't just dropped silently -- `generic_credential_attempts` surfaces them separately, by how many distinct IPs attempted each one. Useful for sanity-checking the denylist itself (does it look like real weak-credential/protocol-noise traffic, or did something legitimate get excluded?).

In [ ]:
from honeypot.analysis.network.charts import plot_generic_credential_breakdown
from honeypot.analysis.network.graph import generic_credential_attempts

generic_attempts = generic_credential_attempts(events)
Image(plot_generic_credential_breakdown(generic_attempts))